# Fake reviwe Detection

1. Dataset

we have the original review datset from on a publicly available source now we will use T5 Model to generate the fake review and shuffle oth the datasets and store them as a csv file

In [ ]:
import pandas as pd
import numpy as np
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch
from tqdm import tqdm
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

def load_t5_model():
    """Load T5 model with error handling"""
    try:
        # Use smaller model for faster execution
        model_name = 't5-small'
        tokenizer = T5Tokenizer.from_pretrained(model_name)
        model = T5ForConditionalGeneration.from_pretrained(model_name)

        # Move to GPU if available
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)

        return model, tokenizer, device
    except Exception as e:
        print(f"Error loading model: {e}")
        raise

# Load dataset
try:
    original_df = pd.read_csv('test.csv')
    original_df['label'] = 'OG'  # Original reviews
except FileNotFoundError:
    print("Error: 'test.csv' not found in current directory")
    raise

# Initialize T5 model
try:
    model, tokenizer, device = load_t5_model()
except:
    print("Failed to initialize T5 model")
    raise

def generate_t5_review(product_id, original_df, max_length=80):
    """Generate a fake review using T5 model with error handling"""
    try:
        # Get product context
        product_reviews = original_df[original_df['item_id'] == product_id]['review']
        example_review = product_reviews.sample(1).iloc[0] if len(product_reviews) > 0 else "a product"

        input_text = f"generate a fake product review similar to: {example_review[:200]}"  # Limit context length

        inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_length=max_length,
                do_sample=True,
                temperature=0.9,  # Higher temperature for more creativity
                top_k=50,
                top_p=0.95,
                num_return_sequences=1
            )

        review = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Post-processing
        review = review.split('.')[0] + '.'
        review = review.replace('"', '').strip()

        return review if len(review) > 10 else "Excellent product, works perfectly."

    except Exception as e:
        print(f"Error generating review for product {product_id}: {e}")
        return "Great product, highly recommended."

# Generate fake reviews
fake_reviews = []
unique_products = original_df['item_id'].unique()

try:
    for i, product_id in enumerate(tqdm(unique_products[:len(original_df)], desc="Generating reviews")):
        review_text = generate_t5_review(product_id, original_df)

        fake_reviews.append({
            'qid': len(original_df) + i + 1,
            'item_id': product_id,
            'user_id': f"T5USER{i+1:04d}",
            'rating': min(5, max(1, int(np.random.normal(4.2, 0.8)))),
            'review': review_text,
            'label': 'CG'  # Computer Generated
        })
except Exception as e:
    print(f"Error during review generation: {e}")

# Create DataFrame and combine
if fake_reviews:
    fake_df = pd.DataFrame(fake_reviews)
    combined_df = pd.concat([original_df, fake_df])

    # Shuffle the dataset
    combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

    # Save to CSV
    combined_df.to_csv('t5_generated_reviews.csv', index=False)
    print(f"Successfully generated dataset with {len(original_df)} real and {len(fake_df)} fake reviews")
else:
    print("Failed to generate any fake reviews")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generating reviews:   1%|▏         | 278/20463 [06:02<7:44:38,  1.38s/it]

In [ ]:
!pip install pandas numpy torch transformers tqdm

In [ ]:
!pip install torch --extra-index-url https://download.pytorch.org/whl/cu117

In [ ]:
!pip install torch==2.0.1 transformers==4.30.2 pandas numpy tqdm

2. Load the requied modules

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
import re

3. Data Preprocessing
now we will load the data and perform the EDA

In [ ]:
data = pd.read_csv("/content/t5_generated_reviews.csv")

In [ ]:
data.head(10)

In [ ]:
ax = data['rating'].value_counts().sort_index() \
    .plot(kind='bar',
          title='Count of Reviews by Stars',
          figsize=(10, 5))
ax.set_xlabel('Review Stars')
plt.show()

In [ ]:
data.shape

In [ ]:
print(data.info())
print(data.describe())
print(data.isnull().sum())


In [ ]:
sns.countplot(x='label', data=data)
plt.title('Distribution of Fake(CG) and Real(OG) Reviews')
plt.show()

print(data['label'].value_counts(normalize=True))


In [ ]:

sns.histplot(data=data, x=data['review'].apply(lambda x: len(str(x).split())), hue='label', bins=50, kde=True)
plt.title('Review Length Distribution by Label')

plt.show()


All the feature in our dataset are very important for making business decisions but for this project we need only specific feature do we will drop the other un necessary ones

In [ ]:
#droping the unwanted features
df = data.drop(columns=['item_id','qid', 'user_id', 'review_length', 'rating'], errors='ignore')
df.head()

In [ ]:
import nltk
nltk.download('stopwords')
stemmer = nltk.SnowballStemmer("english")
from nltk.corpus import stopwords
import string
stopword=set(stopwords.words('english'))

def text_cleaning(text):
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    text = [word for word in text.split(' ') if word not in stopword]
    text=" ".join(text)
    text = [stemmer.stem(word) for word in text.split(' ')]
    text=" ".join(text)
    return text

In [ ]:
df['review'] = df["review"].apply(text_cleaning)

In [ ]:
df.head(20)

In [ ]:
new_data = df.head(41686)
new_data = new_data.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:

new_data.to_csv("cleandata.csv", index=False)


In [ ]:
cleandata = pd.read_csv("cleandata.csv")
cleandata.shape

In [ ]:
cleandata['id'] = range(1, len(cleandata) + 1)
cleandata.head(10)

In [ ]:
print(cleandata.isnull().sum())

In [ ]:
cleandata.dropna(inplace=True)


In [ ]:
print(cleandata.isnull().sum())


In [ ]:
cleandata.head(10)

# **Spliting the data**

In [ ]:
x = cleandata["review"]
y = cleandata["label"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorization = TfidfVectorizer()
xv_train_np = vectorization.fit_transform(x_train)
xv_test_np = vectorization.transform(x_test)


# **Building the Models**

**1. Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
LR = LogisticRegression()
#LR.fit(xv_train, y_train)
LR.fit(xv_train_np, y_train)

In [ ]:
#LR.score(xv_test, y_test)
LR.score(xv_test_np, y_test)

In [ ]:
#pred_LR = LR.predict(xv_test)
pred_LR = LR.predict(xv_test_np)

In [ ]:
print(classification_report(y_test, pred_LR))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

#Calculate the confusion matrix
cm = confusion_matrix(y_test, pred_LR)

#Create a heatmap to visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix for Logistic Regression Classifier")
plt.show()

**2. Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
DT = DecisionTreeClassifier()
#DT.fit(xv_train, y_train)
DT.fit(xv_train_np, y_train)

In [ ]:
#pred_dt = DT.predict(xv_test)
pred_dt = DT.predict(xv_test_np)

In [ ]:
#DT.score(xv_test, y_test)
DT.score(xv_test_np, y_test)

In [ ]:
print(classification_report(y_test, pred_dt))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

#Calculate the confusion matrix
cm = confusion_matrix(y_test, pred_dt)

#Create a heatmap to visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix for Decision Tree Classifier")
plt.show()

**3. Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
RFC = RandomForestClassifier(random_state=0)
RFC.fit(xv_train_np, y_train)

In [ ]:
pred_rfc = RFC.predict(xv_test_np)

In [ ]:
RFC.score(xv_test_np, y_test)

In [ ]:
print(classification_report(y_test, pred_rfc))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

#Calculate the confusion matrix
cm = confusion_matrix(y_test, pred_rfc)

#Create a heatmap to visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix for Random Forest Classifier")
plt.show()

**4. Support Vector Machine**

In [ ]:
from sklearn.svm import SVC

In [ ]:
SVM = SVC(kernel='linear', random_state=0)
SVM.fit(xv_train_np, y_train)

In [ ]:
pred_svm = SVM.predict(xv_test_np)

In [ ]:
SVM.score(xv_test_np, y_test)

In [ ]:
print(classification_report(y_test, pred_svm))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Calculate the confusion matrix
cm = confusion_matrix(y_test, pred_svm)

# Create a heatmap to visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix for Support Vector Machine Classifier")
plt.show()

**5. K Nearest Neighbour**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
KNN = KNeighborsClassifier(n_neighbors=5)
KNN.fit(xv_train_np, y_train)

In [ ]:
pred_knn = KNN.predict(xv_test_np)

In [ ]:
KNN.score(xv_test_np, y_test)

In [ ]:
print(classification_report(y_test, pred_knn))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
#Calculate the confusion matrix
cm = confusion_matrix(y_test, pred_knn)

#Create a heatmap to visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix for k-Nearest Neighbors Classifier")
plt.show()

**6. Naive Bayes Classifier**

In [ ]:
#from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB


In [ ]:
NB = MultinomialNB()
NB.fit(xv_train_np, y_train)
#NB = GaussianNB()
#NB.fit(xv_train_np, y_train)

In [ ]:
pred_nb = NB.predict(xv_test_np)

In [ ]:
NB.score(xv_test_np, y_test)

In [ ]:
print(classification_report(y_test, pred_nb))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
#Calculate the confusion matrix
cm = confusion_matrix(y_test, pred_nb)

#Create a heatmap to visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix for Naive Bayes Classifier")
plt.show()

**7. XG BOOST**

In [ ]:
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# Convert string labels to numeric values
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# Now fit the model using the numeric labels
XG = xgb.XGBClassifier()
XG.fit(xv_train_np, y_train_encoded)  # Use numeric labels


In [ ]:
pred_xg = XG.predict(xv_test_np)

In [ ]:
y_test_encoded = label_encoder.transform(y_test)


In [ ]:
XG.score(xv_test_np, y_test_encoded)

In [ ]:
print(classification_report(y_test_encoded, pred_xg))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Calculate the confusion matrix
cm = confusion_matrix(y_test_encoded, pred_xg)

# Create a heatmap to visualize the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix for XGBoost")
plt.show()

In [ ]:
Now we will select the best model from all the seven model we have trained

In [ ]:

def evaluate_model(name, y_true, y_pred, results, use_encoded=False):
    pos = 1 if use_encoded else 'OG'
    results[name] = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, pos_label=pos, average='binary'),
        "Recall": recall_score(y_true, y_pred, pos_label=pos, average='binary'),
        "F1 Score": f1_score(y_true, y_pred, pos_label=pos, average='binary')
    }

#Collecting results for all models
results = {}
evaluate_model("Logistic Regression", y_test, pred_LR, results)
evaluate_model("Decision Tree", y_test, pred_dt, results)
evaluate_model("Random Forest", y_test, pred_rfc, results)
evaluate_model("SVM", y_test, pred_svm, results)
evaluate_model("KNN", y_test, pred_knn, results)
evaluate_model("Naive Bayes", y_test, pred_nb, results)
evaluate_model("XGBoost", y_test_encoded, pred_xg, results, use_encoded=True)

# Display results
results_df = pd.DataFrame(results).T
print("Model Comparison:")
print(results_df.sort_values("F1 Score"))

Now select the best model

In [ ]:

best_model_name = results_df["F1 Score"].idxmax()
print(f"Best-performing model based on F1 Score: {best_model_name}")

We will undestand and anayse the report and results of the best model

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

#Predict probabilities with XGBoost
if hasattr(XG, "predict_proba"):
    y_prob = XG.predict_proba(xv_test_np)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_encoded, y_prob, pos_label=1)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc_score(y_test_encoded, y_prob):.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("ROC Curve cannot be plotted for XGBoost (no predict_proba method).")


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, _ = precision_recall_curve(y_test_encoded, y_prob)
ap = average_precision_score(y_test_encoded, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'AP = {ap:.2f}')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - XGBoost")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(x_train)  # original text data

# Get feature importance from XGBoost
importances = XG.feature_importances_
features = vectorizer.get_feature_names_out()

# Combine and sort
top_features = sorted(zip(importances, features), reverse=True)[:15]

# Bar plot
importance_vals, feature_names = zip(*top_features)
plt.barh(feature_names, importance_vals)
plt.xlabel("Importance")
plt.title("Top TF-IDF Features in XGBoost")
plt.gca().invert_yaxis()
plt.show()


XGBoost emerged as the best performing model with an outstanding ROC(Receiver operating characteristic curve)-AUC(Area under the curve) and Average Precision score of 0.97, demonstrating excellent classification ability. It achieved high values across all key metrics Accuracy (90.5%), Precision (90.7%), Recall (90.6%), and F1 Score (90.6%), indicating a well-balanced model. The top influential TF-IDF features like "product", "recommend", and "fake" offer valuable interpretability into what drives predictions.


In [ ]:
from wordcloud import WordCloud

fake_text = " ".join(x_train[y_train == 'CG'])
real_text = " ".join(x_train[y_train == 'OG'])

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(WordCloud(width=600, height=400).generate(fake_text))
plt.title("Fake Reviews (CG)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(WordCloud(width=600, height=400).generate(real_text))
plt.title("Original Reviews (OG)")
plt.axis('off')

plt.tight_layout()
plt.show()


Manual Testing (optional)

In [ ]:
# Manual testing function
def manual_test(review):
    review_cleaned = text_cleaning(review)
    vector_input = vectorization.transform([review_cleaned])
    prediction = XG.predict(vector_input)
    result = "Fake Review" if prediction[0] == 1 else "Genuine Review"
    return result



In [ ]:
review = input()

print("Review 1:", manual_test(review))
